Import the relevant python modules required for the example: xarray and matplotlib

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from SubRoutines.plotting import pannel_plot, pannel_hist, LccsManager

Read in the dataset

In [ ]:
esacci_lst_025 = "Data/ESACCI-LST-L3C-LST-MODISA-0.25deg_1MONTHLY_DAY-20060701000000-fv3.00.nc"
ds = xr.load_dataset(esacci_lst_025)

Display the contents of the dataset

In [ ]:
ds

We can plot the total uncertainty in LST across the globe.  The largest uncertainties can be seen in regions where data coverage is limited due to cloud cover, including the Saharan Desert and across India.  Total cloud cover can be seasonal so the areas of largest uncertainty may vary.

In [ ]:
ds.lst_uncertainty.plot(size=8,aspect=2,robust=True)

We can summarise the global fields by latitude using the zonal mean, the mean over all longitudes. Note that the right-hand plot is the average uncertainty of an individual pixel at each latitude, so it shows how reliable a typical LST value is. It is not the uncertainty of the zonal mean itself, because averaging many pixels reduces the effect of uncorrelated errors.

In [ ]:
zonal_lst = ds.lst.mean("lon").squeeze()
zonal_unc = ds.lst_uncertainty.mean("lon").squeeze()

fig, ax = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
zonal_lst.plot(y="lat", ax=ax[0])
ax[0].set_title("Zonal mean LST")
zonal_unc.plot(y="lat", ax=ax[1])
ax[1].set_title("Zonal mean of per-pixel total uncertainty")

A plain mean over the grid gives every pixel equal weight, but grid cells get smaller towards the poles. Weighting by the cosine of latitude, using xarray's `.weighted`, corrects for this and gives a different answer:

In [ ]:
weights = np.cos(np.deg2rad(ds.lat))
mean_plain = ds.lst.mean().item()
mean_weighted = ds.lst.weighted(weights).mean().item()
print(f"Mean LST over land pixels with data - unweighted: {mean_plain:.1f} K, area-weighted: {mean_weighted:.1f} K")

Next we can plot a smaller region of the globe to look at the uncertainties in more detail.  Zooming in on this region over China and Japan we can see that some surface features appear in the uncertainty data.  This is because there is a component of the uncertainty that is related to our representation of the land cover.  There are also larger uncertainties along the coast, particularly evident around the islands where there are fewer satellite observations over land within these grid boxes (some are neighbouring ocean pixels) so there are less data going into the LST estimate.  The panel plots below show the different uncertainty components that form the total uncertainty.  The surface features such as rivers and high ground are evident in the bottom right plot.  The larger uncertainties along the coast are split between the surface, atmosphere and random components (note that the colourbar scales are different for each component).  Random uncertainties include instrument noise and sampling - these are uncertainties that are uncorrelated between neighbouring LST values.  Try changing the region by editing the code above so that you can see how the uncertainties vary in the part of the world you are most interested in.

In [ ]:
lower_lon = 105.0
upper_lon = 145.0
lower_lat = 25.0
upper_lat = 65.0
region = ds.sel(lat=slice(lower_lat, upper_lat), lon=slice(lower_lon, upper_lon))

In [ ]:
region.lst_uncertainty.plot(size=8,aspect=2, robust=True)

In [ ]:
pannel_plot(region)

We can plot a histogram of the total uncertainties over this region and alhtough there are one or two larger values, we find that the uncertainties typically range between 0.5-1 K.

In [ ]:
region.lst_uncertainty.plot.hist(size=8,bins=50, alpha=0.5, color='steelblue',edgecolor='grey')
;

Similarly we can plot histograms of the different uncertainty components.  In this particular region the largest uncertainties are arising from the surface component.

In [ ]:
pannel_hist(region)

To see the different uncertainty contributions and how they propogate we will manually calculate the total uncertainty from the components. First we assign a point to do the calculation for:

In [ ]:
lat_point, lon_point = 35.0, 115.0

point = ds.sel(lat=lat_point, lon=lon_point, method="nearest")

Then we extract the total uncertainty and the uncertainty components for that point:

In [ ]:
point_total_uncertainty = point.lst_uncertainty.values[0]

ran = point.lst_unc_ran.values[0]
atm = point.lst_unc_loc_atm.values[0]
sfc = point.lst_unc_loc_sfc.values[0]
sys = point.lst_unc_sys.values[0]
print(f"Total: {point_total_uncertainty:.3f}"[:-1])
print(f"random: {ran:.3f}, atmosphere: {atm:.3f}, surface: {sfc:.3f}, systematic: {sys:3f}")

To combine the components to make the total uncertainty they are added in quadrature:

In [ ]:
calculated_total = np.sqrt(ran**2 + atm**2 + sfc**2 + sys**2)
print(f"Calculated total: {calculated_total:.3f}"[:-1])

The uncertainty can be used to give context to the LST in terms of the degree to which we have confidence in the LST value. We can visualise this on a LST transect:

In [ ]:
lat_transect = 30.0
lower_lon = 5.0
upper_lon = 9.0

transect = ds.sel(lat=lat_transect, method="nearest").sel(lon=slice(lower_lon, upper_lon)).squeeze()

Now we can plot the LST with a region bounded by the uncertainty shown:

In [ ]:
upper_bound = transect.lst.values+transect.lst_uncertainty.values
lower_bound = transect.lst.values-transect.lst_uncertainty.values

fig, ax = plt.subplots(figsize=(16,8))
transect.lst.plot(ax=ax)
ax.fill_between(transect.lon.values,upper_bound,lower_bound,alpha=0.2)


The uncertainty can also be used to screen the data. Here we keep only the pixels where the total uncertainty is below a chosen threshold (try changing it), and check how much of the data is retained. Pixels with no uncertainty value are also removed:

In [ ]:
threshold = 1.0  # K
good = ds.lst.where(ds.lst_uncertainty < threshold)

kept = good.notnull().sum().item() / ds.lst.notnull().sum().item()
print(f"Pixels kept with uncertainty < {threshold} K: {kept:.1%}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 5), sharey=True)
vmin, vmax = ds.lst.min().item(), ds.lst.max().item()
ds.lst.squeeze().plot(ax=ax[0], vmin=vmin, vmax=vmax)
ax[0].set_title("All LST")
good.squeeze().plot(ax=ax[1], vmin=vmin, vmax=vmax)
ax[1].set_title(f"LST with uncertainty < {threshold} K")

In [ ]:
esacci_lst_001 = "Data/ESACCI-LST-L3C-LST-MODISA-0.01deg_1MONTHLY_DAY-20220901103756-fv3.00.nc"
regridded_ds = xr.load_dataset(esacci_lst_001)

In [ ]:
lccs_data = LccsManager(regridded_ds)
lccs_data.analyse_region()
lccs_data.bin_sfc_unc_by_lcc()
lccs_data.unc_box_plot()

The 0.01 degree file also contains the number of clear-sky observations (`n`) and the variance of the observations that went into each monthly value:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 6))
regridded_ds.n.squeeze().plot(ax=ax[0], robust=True)
ax[0].set_title("Number of clear-sky observations (n)")
regridded_ds.variance.squeeze().plot(ax=ax[1], robust=True)
ax[1].set_title("Variance of the observations (K$^2$)")

We can compare the number of clear-sky observations with an uncertainty component. In this region the atmospheric component tends to be lower where there are more clear-sky observations, but the other components (and so the total) do not show the same pattern. Try plotting the other `lst_unc_*` variables:

In [ ]:
n = regridded_ds.n.squeeze()
atm_unc = regridded_ds.lst_unc_loc_atm.squeeze()
valid = (n.notnull() & atm_unc.notnull()).values

fig, ax = plt.subplots(figsize=(8, 6))
hb = ax.hexbin(n.values[valid], atm_unc.values[valid], gridsize=30, mincnt=1, bins="log")
fig.colorbar(hb, label="Number of pixels (log scale)")
ax.set_xlabel("Number of clear-sky observations (n)")
ax.set_ylabel("Locally correlated atmospheric uncertainty (K)")